In [ ]:
from cae import *

data_root = "./data"
techniques = ("sliding_window", "resize")
losses = ("SSIM+MSE", "MSE")

device = "cpu"
if torch.cuda.is_available():
    device = torch.device("cuda:0")
print(f"Running on device {device}")

for technique in techniques:
    for loss in losses:
        experiment_id = technique + "_cae_" + loss + "-loss"
        weights_path = os.path.join("./results", experiment_id, "cae.pt")
        if not os.path.exists(weights_path):
            run_cae_experiment(data_root, loss, technique)  # will create weight file

        model = ConvolutionalAutoEncoder(5)
        model.load_state_dict(torch.load(weights_path, map_location=device))
        model.to(device)

        plot_loss(experiment_id)

        print(f"Results for {experiment_id}")

        if technique == "sliding_window":
            resize_res = (2048, 2304)
            batch_size = 1
        elif technique == "resize":
            resize_res = (256, 256)
            batch_size = 32

        transforms = v2.Compose(
            [v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Resize(resize_res)]
        )

        test_set = FabricDataset(
            root=data_root,
            split="test",
            transforms=transforms,
            target_res=resize_res,
        )

        test_loader = DataLoader(
            dataset=test_set,
            batch_size=batch_size,
            shuffle=False,
            num_workers=4,
            pin_memory=True,
        )

        test_scores, test_labels = evaluate(model, test_loader, device, technique)
        print(
            f"Image-Level AUROC: {roc_auc_score(test_labels, test_scores):.4f} "
            f"PR-AUC: {average_precision_score(test_labels, test_scores):.4f}",
            flush=True,
        )

        model.eval()
        with torch.no_grad():
            # Select an example from the test set (last are anomalies)
            image, label, gt_mask = test_set[-1]

            # Add batch dimension and move to device
            image_batch = image.unsqueeze(0).to(device)

            # Get reconstruction
            if technique == "sliding_window":
                reconstruction_batch = sliding_window_inference(model, image_batch)
            elif technique == "resize":
                reconstruction_batch = model(image_batch)

            # Compute element-wise MSE for anomaly map
            mse_elementwise = torch.nn.MSELoss(reduction="none")
            error_map = mse_elementwise(image_batch, reconstruction_batch)
            anomaly_map = torch.mean(error_map, dim=1).squeeze(
                0
            )  # Average across channels, remove batch dim

            # Plot the results
            print(f"Displaying example: Label = {label} (0=good, 1=bad)")
            plot_anomaly_results(
                original_image=image,
                reconstruction=reconstruction_batch.squeeze(0).cpu(),
                anomaly_map=anomaly_map,
                gt_mask=gt_mask,
            )